In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import re
from datetime import datetime

def extract_dates(text):
    """
    Extract dates in various formats:
    - MM/DD/YYYY
    - DD-MM-YYYY
    - Month DD, YYYY
    - YYYY-MM-DD (ISO)
    """
    
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',  # MM/DD/YYYY
        r'\d{1,2}-\d{1,2}-\d{4}',  # DD-MM-YYYY
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}',  # Month DD, YYYY
        r'\d{4}-\d{2}-\d{2}'  # ISO format
    ]
    
    dates = []
    
    for pattern in patterns:
        matches = re.findall(pattern, text)
        dates.extend(matches)
    
    return dates


# Test
text = "Invoice date: 03/15/2024. Due: March 30, 2024"
print(extract_dates(text))

['03/15/2024', 'March 30, 2024']


In [2]:
import re

def extract_amounts(text):
    """
    Extract currency amounts.
    Handles:
    - $1,250.50
    - 1250.50
    - $1250
    """
    
    pattern = r'\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'
    amounts = re.findall(pattern, text)
    
    cleaned = []
    
    for amount in amounts:
        # Remove $ and commas
        clean = amount.replace('$', '').replace(',', '')
        
        try:
            cleaned.append(float(clean))
        except ValueError:
            continue  # Skip invalid conversions
    
    return cleaned


# Test
text = "Total: $1,250.50. Tax: $125.05. Subtotal: 1125.45"
print(extract_amounts(text))

[1250.5, 125.05, 112.0, 5.45]


In [3]:
import re

def extract_invoice_number(text):
    """
    Extract invoice/order numbers.
    Handles:
    - INV-2024-001
    - #12345
    - ORDER-ABC123
    - Invoice Number: XXX
    """
    
    patterns = [
        r'(INV-\d{4}-\d{3})',
        r'(#\d{5,})',
        r'(ORDER-[A-Z0-9]+)',
        r'Invoice (?:Number|#):?\s*([A-Z0-9-]+)'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1)
    
    return None


# Test
text = "Invoice Number: INV-2024-001"
print(extract_invoice_number(text))

INV-2024-001


In [4]:
import spacy

# Load model (with fallback for Kaggle)
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    # Install model if not present
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"], check=True)
    nlp = spacy.load("en_core_web_sm")


# Sample invoice text
text = """
Invoice from Acme Corporation
123 Main Street, New York, NY 10001
Contact: John Smith (john@acme.com)
Date: March 15, 2024
Amount Due: $1,250.50
"""

# Process text
doc = nlp(text)

# Extract entities
print("Found entities:\n")
for ent in doc.ents:
    print(f"{ent.text:25} {ent.label_:10} {spacy.explain(ent.label_)}")

Found entities:

Acme Corporation          ORG        Companies, agencies, institutions, etc.
123                       CARDINAL   Numerals that do not fall under another type
Main Street               FAC        Buildings, airports, highways, bridges, etc.
New York                  GPE        Countries, cities, states
10001                     DATE       Absolute or relative dates or periods
John Smith                PERSON     People, including fictional
March 15, 2024            DATE       Absolute or relative dates or periods
1,250.50                  MONEY      Monetary values, including unit


In [5]:
import spacy

# Ensure model is loaded (reuse if already loaded)
try:
    nlp
except NameError:
    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError:
        import subprocess
        subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"], check=True)
        nlp = spacy.load("en_core_web_sm")


def extract_entities(text):
    """
    Extract and organize entities by type.
    """
    doc = nlp(text)

    entities = {
        'persons': [],
        'organizations': [],
        'locations': [],
        'dates': [],
        'money': []
    }

    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities['persons'].append(ent.text)
        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text)
        elif ent.label_ in ['GPE', 'LOC']:
            entities['locations'].append(ent.text)
        elif ent.label_ == 'DATE':
            entities['dates'].append(ent.text)
        elif ent.label_ == 'MONEY':
            entities['money'].append(ent.text)

    # Optional: remove duplicates
    for key in entities:
        entities[key] = list(set(entities[key]))

    return entities


# Test
text = """
Invoice from Acme Corporation
123 Main Street, New York, NY 10001
Contact: John Smith (john@acme.com)
Date: March 15, 2024
Amount Due: $1,250.50
"""

result = extract_entities(text)

for entity_type, values in result.items():
    print(f"{entity_type}: {values}")

persons: ['John Smith']
organizations: ['Acme Corporation']
locations: ['New York']
dates: ['March 15, 2024', '10001']
money: ['1,250.50']


In [9]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")

text = """
Invoice from Acme Corporation
123 Main Street, New York, NY 10001
Contact: John Smith (john@acme.com)
Date: March 15, 2024
Amount Due: $1,250.50
"""

doc = nlp(text)

# Force rendering via manual template (robust)
from spacy.tokens import Doc
from spacy import displacy

html = displacy.render([doc], style="ent", page=True)

# Hard fallback if still None
if html is None:
    html = "<html><body><h3>Entities</h3><ul>"
    for ent in doc.ents:
        html += f"<li>{ent.text} ({ent.label_})</li>"
    html += "</ul></body></html>"

# Save
with open("entities.html", "w", encoding="utf-8") as f:
    f.write(html)

print("Saved successfully")

Saved successfully


In [11]:
import json
import re
import spacy
import pytesseract
from PIL import Image

# ---------- Load spaCy model safely ----------
try:
    nlp
except NameError:
    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError:
        import subprocess
        subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"], check=True)
        nlp = spacy.load("en_core_web_sm")


# ---------- Regex Extractors ----------
def extract_dates(text):
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',
        r'\d{1,2}-\d{1,2}-\d{4}',
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}',
        r'\d{4}-\d{2}-\d{2}'
    ]
    dates = []
    for p in patterns:
        dates.extend(re.findall(p, text))
    return dates


def extract_amounts(text):
    pattern = r'\$?\d+(?:,\d{3})*(?:\.\d{2})?'
    matches = re.findall(pattern, text)

    amounts = []
    for m in matches:
        try:
            clean = m.replace('$', '').replace(',', '')
            amounts.append(float(clean))
        except:
            continue
    return amounts


def extract_invoice_number(text):
    patterns = [
        r'(INV-\d{4}-\d{3})',
        r'(#\d{5,})',
        r'(ORDER-[A-Z0-9]+)',
        r'Invoice (?:Number|#):?\s*([A-Z0-9-]+)'
    ]

    for p in patterns:
        match = re.search(p, text, re.IGNORECASE)
        if match:
            return match.group(1)
    return None


# ---------- NER Extractor ----------
def extract_entities(text):
    doc = nlp(text)

    entities = {
        'persons': [],
        'organizations': [],
        'locations': [],
        'dates_ner': [],
        'money_ner': []
    }

    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities['persons'].append(ent.text)
        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text)
        elif ent.label_ in ['GPE', 'LOC']:
            entities['locations'].append(ent.text)
        elif ent.label_ == 'DATE':
            entities['dates_ner'].append(ent.text)
        elif ent.label_ == 'MONEY':
            entities['money_ner'].append(ent.text)

    # remove duplicates
    for k in entities:
        entities[k] = list(set(entities[k]))

    return entities


# ---------- Main Pipeline ----------
def process_invoice(image_path):
    """
    OCR → Regex Extraction → NER → Structured JSON
    """

    # Step 1: OCR
    try:
        img = Image.open(image_path)
        text = pytesseract.image_to_string(img)
    except Exception as e:
        return {"error": f"OCR failed: {str(e)}"}

    # Step 2: Regex extraction
    invoice_data = {
        'raw_text': text,
        'invoice_number': extract_invoice_number(text),
        'dates': extract_dates(text),
        'amounts': extract_amounts(text)
    }

    # Step 3: NER
    entities = extract_entities(text)
    invoice_data.update(entities)

    # Step 4: Post-processing
    if invoice_data['amounts']:
        invoice_data['total_amount'] = max(invoice_data['amounts'])
    else:
        invoice_data['total_amount'] = None

    if invoice_data['dates']:
        invoice_data['invoice_date'] = invoice_data['dates'][0]
    else:
        invoice_data['invoice_date'] = None

    return invoice_data


# ---------- Test ----------
result = process_invoice("/kaggle/input/datasets/ammaraakhtar11/ocrextraction/images/13.jpg")
print(json.dumps(result, indent=2))

{
  "raw_text": " \n\nTD A ISWOYBKCH\n\n \n\nWalmart >\\<\n\nSave money. Live better.\n\n( 843) 292 - 0962\nMANAGER MICHAEL (\u00a9 [GHARD\n2014 S TRY ST\n\n \n\n  \n\nFLORENCE $0. 23506,\nS1# 02703 OPH 009049 TE 49 1RA Or42\nGIN CARO 087458604333 50.00 0\n\u2018SUBIOL A 50.00\nTOTAL 50. (0,\n\nCASH TN) 60.00\nCHANGE UE 10.06\n\nSHOP.CARL ACTIVATION. 80,00\nACCOUNT 613968545249/29\"\nAPPR. CODE \u2014 792986.\nREF douo7A2\n\nBal Tran Amt End Bal\n0.00 50.09) 50.00\nO/T 17 1\n\n   \n\n92\n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\nTANTATAAT\n\nTCH 9796 5895 92387 9333 1992\n|\nLow Prices You Cat Trust\nO8/AT\nStore reseipts o7 Your phone. Walmart P\n\n  \n\n \n\n \n\n \n\f",
  "invoice_number": null,
  "dates": [],
  "amounts": [
    843.0,
    292.0,
    962.0,
    2014.0,
    0.0,
    23506.0,
    1.0,
    2703.0,
    9049.0,
    49.0,
    1.0,
    42.0,
    87458604333.0,
    50.0,
    0.0,
    50.0,
    50.0,
    0.0,
    60.0,
    10.06,
    80.0,
    0.0,
 

In [12]:
# Save to JSON file
output_file = "extracted_data.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2)

print(f"Results saved to {output_file}")

Results saved to extracted_data.json
